In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
from ast import increment_lineno
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')
%matplotlib inline

import tensorflow as tf
from keras.preprocessing.image import load_img
from keras.models import Sequential, Model
from keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D, Input

In [5]:
import numpy as np

X = np.load('/content/drive/MyDrive/face analyser data/X.npy')
age_labels = np.load('/content/drive/MyDrive/face analyser data/age_labels.npy')
gender_labels = np.load('/content/drive/MyDrive/face analyser data/gender_labels.npy')

print(X.shape, X.dtype)
print(age_labels.shape, gender_labels.shape)

(23708, 128, 128, 3) float32
(23708,) (23708,)


In [6]:
X.shape

(23708, 128, 128, 3)

In [7]:
X = X/255.0

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_val, age_train, age_val, gender_train, gender_val = train_test_split(
    X, age_labels, gender_labels, test_size=0.2, random_state=42
)

del X
import gc
gc.collect()

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape  : {X_val.shape}")

X_train shape: (18966, 128, 128, 3)
X_val shape  : (4742, 128, 128, 3)


In [9]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

input_layer = Input(shape=(128, 128, 3))

# Shared CNN layers
x = Conv2D(32, (3,3), activation='relu', padding='same')(input_layer)
x = BatchNormalization()(x)
x = MaxPooling2D()(x)

x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D()(x)

x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D()(x)

x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)

# Two output heads
age_output    = Dense(1, name='age')(x)
gender_output = Dense(1, activation='sigmoid', name='gender')(x)

model = Model(inputs=input_layer, outputs=[age_output, gender_output])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss={
        'age'   : 'mse',
        'gender': 'binary_crossentropy'
    },
    metrics={
        'age'   : 'mae',
        'gender': 'accuracy'
    }
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128, 128,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 16, 16,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 32768)     │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │  8,388,864 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ age (Dense)         │ (None, 1)         │        257 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gender (Dense)      │ (None, 1)         │        257 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,483,522 (32.36 MB)

 Trainable params: 8,483,074 (32.36 MB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from google.colab import drive

# Drive mount karo model save karne ke liye
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/FaceAnalyser/models', exist_ok=True)

checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/FaceAnalyser/models/face_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

model.fit(
    X_train,
    {'age': age_train, 'gender': gender_train},
    validation_data=(X_val, {'age': age_val, 'gender': gender_val}),
    epochs=20,
    batch_size=32,
    callbacks=[checkpoint, early_stop]
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import gc
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from google.colab import drive

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/face analyser data'

# Load labels only
age_labels    = np.load(f'{BASE}/age_labels.npy')
gender_labels = np.load(f'{BASE}/gender_labels.npy')

# Split indexes
indices = np.arange(len(age_labels))
np.random.seed(42)
np.random.shuffle(indices)

split     = int(0.8 * len(indices))
train_idx = indices[:split]
val_idx   = indices[split:]

# Labels split
age_train,    age_val    = age_labels[train_idx],    age_labels[val_idx]
gender_train, gender_val = gender_labels[train_idx], gender_labels[val_idx]

del age_labels, gender_labels
gc.collect()

# X mmap se load — RAM mein nahi jaayega
X = np.load(f'{BASE}/X.npy', mmap_mode='r')

# Generator — local indexes use karega
def data_generator(X, X_indices, age_labels, gender_labels, batch_size=32):
    local_age    = age_labels
    local_gender = gender_labels
    idx          = np.arange(len(X_indices))
    while True:
        np.random.shuffle(idx)
        for i in range(0, len(idx), batch_size):
            batch      = idx[i:i+batch_size]
            real_idx   = X_indices[batch]
            X_batch    = X[real_idx].copy()
            yield X_batch, {
                'age'   : local_age[batch],
                'gender': local_gender[batch]
            }

train_gen = data_generator(X, train_idx, age_train, gender_train)
val_gen   = data_generator(X, val_idx,   age_val,   gender_val)

train_steps = len(train_idx) // 32
val_steps   = len(val_idx)   // 32

# Model
input_layer = Input(shape=(128, 128, 3))
x = Conv2D(32,  (3,3), activation='relu', padding='same')(input_layer)
x = BatchNormalization()(x)
x = MaxPooling2D()(x)
x = Conv2D(64,  (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D()(x)
x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = BatchNormalization()(x)
x = MaxPooling2D()(x)
x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)

age_output    = Dense(1, name='age')(x)
gender_output = Dense(1, activation='sigmoid', name='gender')(x)

model = Model(inputs=input_layer, outputs=[age_output, gender_output])
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss={'age': 'mse', 'gender': 'binary_crossentropy'},
    metrics={'age': 'mae', 'gender': 'accuracy'}
)

model.summary()

# Callbacks
os.makedirs(f'{BASE}/models', exist_ok=True)

checkpoint = ModelCheckpoint(
    f'{BASE}/models/face_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Train
model.fit(
    train_gen,
    steps_per_epoch=train_steps,
    validation_data=val_gen,
    validation_steps=val_steps,
    epochs=20,
    callbacks=[checkpoint, early_stop]
)

print("✅ Done!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 128, 128,  │        896 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        128 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │     18,496 │ max_pooling2d_3[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 32, 32,    │     73,856 │ max_pooling2d_4[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        512 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 16, 16,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 32768)     │          0 │ max_pooling2d_5[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │  8,388,864 │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ age (Dense)         │ (None, 1)         │        257 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gender (Dense)      │ (None, 1)         │        257 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,483,522 (32.36 MB)

 Trainable params: 8,483,074 (32.36 MB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/20
591/592 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - age_loss: 436.9094 - age_mae: 15.1632 - gender_accuracy: 0.5852 - gender_loss: 2.6112 - loss: 439.5206
Epoch 1: val_loss improved from None to 198.79688, saving model to /content/drive/MyDrive/face analyser data/models/face_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/face analyser data/models/face_model.keras
592/592 ━━━━━━━━━━━━━━━━━━━━ 53s 76ms/step - age_loss: 283.0448 - age_mae: 12.5123 - gender_accuracy: 0.6150 - gender_loss: 1.6743 - loss: 284.7191 - val_age_loss: 198.1687 - val_age_mae: 10.3142 - val_gender_accuracy: 0.6997 - val_gender_loss: 0.6282 - val_loss: 198.7969
Epoch 2/20
590/592 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - age_loss: 162.9556 - age_mae: 9.5942 - gender_accuracy: 0.6654 - gender_loss: 0.6733 - loss: 163.6329
Epoch 2: val_loss improved from 198.79688 to 114.00378, saving model to /content/drive/MyDrive/face analyser data/models/face_model.keras

Epoch 2: finished saving model to /cont

In [3]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

TensorFlow: 2.20.0
Keras: 3.13.2
